# Delay Risk Prediction

## Objective

Predict whether an EV service visit will be delayed.

### Target
Is_Delayed

### Problem Type
Binary Classification

### Classes

- 0 → Not Delayed
- 1 → Delayed

### Models

- Logistic Regression
- Decision Tree Classifier
- Random Forest Classifier

### Evaluation Metrics

- Accuracy
- Precision
- Recall
- F1 Score
- ROC-AUC

### Important Business Objective

The model should identify service visits that are at risk of delay
before the service is completed, allowing the operations team to
take preventive action.

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)
    

In [6]:
df = pd.read_csv("ml_ready.csv")
df.shape

(6583, 26)

In [8]:
print(df["Is_Delayed"].value_counts())

Is_Delayed
False    5721
True      862
Name: count, dtype: int64


In [7]:
print(
    df["Is_Delayed"]
    .value_counts(normalize=True)
    .round(3)
)

Is_Delayed
False    0.869
True     0.131
Name: proportion, dtype: float64


In [9]:
X = df.drop(
    columns=[
        "Repair_Cost",
        "Turnaround_Time_Days",
        "Is_Delayed"
    ]
)

y = df["Is_Delayed"]

In [10]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (6583, 23)
y shape: (6583,)


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [12]:
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).round(3))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True).round(3))

X_train: (5266, 23)
X_test : (1317, 23)

Training target distribution:
Is_Delayed
False    0.869
True     0.131
Name: proportion, dtype: float64

Test target distribution:
Is_Delayed
False    0.869
True     0.131
Name: proportion, dtype: float64


In [13]:
numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "bool"]
).columns.tolist()

print("Numerical Features:", len(numerical_features))
print(numerical_features)

print("\nCategorical Features:", len(categorical_features))
print(categorical_features)

Numerical Features: 13
['Visit_Number', 'Vehicle_Age_at_Service', 'Battery_Age_at_Service', 'Battery_Health_at_Service', 'Expected_Part_ETA_Days', 'Active_Jobs_On_Arrival', 'Workshop_Utilization', 'Technician_Experience_Years', 'Repair_Complexity', 'Base_Labor_Hours', 'Technician_Efficiency', 'Effective_Labor_Hours', 'Expected_TAT_Days']

Categorical Features: 10
['Vehicle_Model', 'Battery_Replaced', 'Issue_Family', 'Exact_Issue', 'Parts_Required', 'Parts_Available', 'Part_Ordered', 'Day_Type', 'Warranty_Status', 'Warranty_Covered']


In [14]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            "passthrough",
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

In [17]:
logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=3000
            )
        )
    ]
)

In [18]:
logistic_model.fit(
    X_train,
    y_train
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  ['Visit_Number',
                                                   'Vehicle_Age_at_Service',
                                                   'Battery_Age_at_Service',
                                                   'Battery_Health_at_Service',
                                                   'Expected_Part_ETA_Days',
                                                   'Active_Jobs_On_Arrival',
                                                   'Workshop_Utilization',
                                                   'Technician_Experience_Years',
                                                   'Repair_Complexity',
                                                   'Base_Labor_Hours',
                                                   'Technician_Efficiency',
                                                   'Effective_Labor_Hours',
                                                   'Expected_TAT_Days']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Vehicle_Model',
                                                   'Battery_Replaced',
                                                   'Issue_Family',
                                                   'Exact_Issue',
                                                   'Parts_Required',
                                                   'Parts_Available',
                                                   'Part_Ordered', 'Day_Type',
                                                   'Warranty_Status',
                                                   'Warranty_Covered'])])),
                ('model', LogisticRegression(max_iter=3000))])

In [19]:
y_pred_logistic = logistic_model.predict(
    X_test
)

In [20]:
y_prob_logistic = logistic_model.predict_proba(
    X_test
)[:, 1]

In [21]:
accuracy_logistic = accuracy_score(
    y_test,
    y_pred_logistic
)

print(f"Accuracy: {accuracy_logistic:.3f}")

Accuracy: 0.892


In [22]:
precision_logistic = precision_score(
    y_test,
    y_pred_logistic
)

recall_logistic = recall_score(
    y_test,
    y_pred_logistic
)

f1_logistic = f1_score(
    y_test,
    y_pred_logistic
)

print(f"Precision: {precision_logistic:.3f}")
print(f"Recall   : {recall_logistic:.3f}")
print(f"F1 Score : {f1_logistic:.3f}")

Precision: 0.708
Recall   : 0.297
F1 Score : 0.418


In [23]:
roc_auc_logistic = roc_auc_score(
    y_test,
    y_prob_logistic
)

print(f"ROC-AUC: {roc_auc_logistic:.3f}")

ROC-AUC: 0.836


In [24]:
cm_logistic = confusion_matrix(
    y_test,
    y_pred_logistic
)

print(cm_logistic)

[[1124   21]
 [ 121   51]]


In [25]:
tree_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            DecisionTreeClassifier(
                random_state=42
            )
        )
    ]
)

In [26]:
tree_model.fit(
    X_train,
    y_train
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  ['Visit_Number',
                                                   'Vehicle_Age_at_Service',
                                                   'Battery_Age_at_Service',
                                                   'Battery_Health_at_Service',
                                                   'Expected_Part_ETA_Days',
                                                   'Active_Jobs_On_Arrival',
                                                   'Workshop_Utilization',
                                                   'Technician_Experience_Years',
                                                   'Repair_Complexity',
                                                   'Base_Labor_Hours',
                                                   'Technician_Efficiency',
                                                   'Effective_Labor_Hours',
                                                   'Expected_TAT_Days']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Vehicle_Model',
                                                   'Battery_Replaced',
                                                   'Issue_Family',
                                                   'Exact_Issue',
                                                   'Parts_Required',
                                                   'Parts_Available',
                                                   'Part_Ordered', 'Day_Type',
                                                   'Warranty_Status',
                                                   'Warranty_Covered'])])),
                ('model', DecisionTreeClassifier(random_state=42))])

In [27]:
y_pred_tree = tree_model.predict(
    X_test
)

y_prob_tree = tree_model.predict_proba(
    X_test
)[:, 1]

In [28]:
accuracy_tree = accuracy_score(
    y_test,
    y_pred_tree
)

precision_tree = precision_score(
    y_test,
    y_pred_tree
)

recall_tree = recall_score(
    y_test,
    y_pred_tree
)

f1_tree = f1_score(
    y_test,
    y_pred_tree
)

roc_auc_tree = roc_auc_score(
    y_test,
    y_prob_tree
)

print(f"Accuracy : {accuracy_tree:.3f}")
print(f"Precision: {precision_tree:.3f}")
print(f"Recall   : {recall_tree:.3f}")
print(f"F1 Score : {f1_tree:.3f}")
print(f"ROC-AUC  : {roc_auc_tree:.3f}")

Accuracy : 0.838
Precision: 0.389
Recall   : 0.419
F1 Score : 0.403
ROC-AUC  : 0.660


In [29]:
cm_tree = confusion_matrix(
    y_test,
    y_pred_tree
)

print(cm_tree)

[[1032  113]
 [ 100   72]]


In [30]:
forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=200,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [31]:
forest_model.fit(
    X_train,
    y_train
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  ['Visit_Number',
                                                   'Vehicle_Age_at_Service',
                                                   'Battery_Age_at_Service',
                                                   'Battery_Health_at_Service',
                                                   'Expected_Part_ETA_Days',
                                                   'Active_Jobs_On_Arrival',
                                                   'Workshop_Utilization',
                                                   'Technician_Experience_Years',
                                                   'Repair_Complexity',
                                                   'Base_Labor_Hours',
                                                   'Technician_Efficiency',
                                                   'Effective_Labor_Hours',
                                                   'Expected_TAT_Days']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Vehicle_Model',
                                                   'Battery_Replaced',
                                                   'Issue_Family',
                                                   'Exact_Issue',
                                                   'Parts_Required',
                                                   'Parts_Available',
                                                   'Part_Ordered', 'Day_Type',
                                                   'Warranty_Status',
                                                   'Warranty_Covered'])])),
                ('model',
                 RandomForestClassifier(n_estimators=200, n_jobs=-1,
                                        random_state=42))])

In [32]:
y_pred_forest = forest_model.predict(
    X_test
)

y_prob_forest = forest_model.predict_proba(
    X_test
)[:, 1]

In [33]:
accuracy_forest = accuracy_score(
    y_test,
    y_pred_forest
)

precision_forest = precision_score(
    y_test,
    y_pred_forest
)

recall_forest = recall_score(
    y_test,
    y_pred_forest
)

f1_forest = f1_score(
    y_test,
    y_pred_forest
)

roc_auc_forest = roc_auc_score(
    y_test,
    y_prob_forest
)

print(f"Accuracy : {accuracy_forest:.3f}")
print(f"Precision: {precision_forest:.3f}")
print(f"Recall   : {recall_forest:.3f}")
print(f"F1 Score : {f1_forest:.3f}")
print(f"ROC-AUC  : {roc_auc_forest:.3f}")

Accuracy : 0.901
Precision: 0.756
Recall   : 0.360
F1 Score : 0.488
ROC-AUC  : 0.845


In [34]:
cm_forest = confusion_matrix(
    y_test,
    y_pred_forest
)

print(cm_forest)

[[1125   20]
 [ 110   62]]
